In [31]:
%%writefile main.py
from fastapi import FastAPI, Depends
from pydantic import BaseModel
from langchain_groq import ChatGroq
import os
from fastapi.responses import StreamingResponse

GROQ_API_KEY = ""

app = FastAPI(title="ReAct Agent API", version="1.0.0")

class AskRequest(BaseModel):
    question: str

class AskResponse(BaseModel):
    answer: str

def get_agent():
    # your Day-7 agent, built once and reused
    return ChatGroq(model="llama-3.3-70b-versatile",
                    api_key=GROQ_API_KEY, temperature=0)

@app.post("/ask", response_model=AskResponse)      # 📍 typed in AND out
async def ask(req: AskRequest, agent=Depends(get_agent)):
    result = agent.invoke(req.question)             # (your ReAct loop here)
    return AskResponse(answer=result.content)

async def token_stream(question: str):
    llm = get_agent()
    # 🌊 .stream() yields chunks as the model produces them
    for chunk in llm.stream(question):
        text = chunk.content
        if text:
            yield f"data: {text}\n\n"      # SSE format: 'data: ...\n\n'
    yield "data: [DONE]\n\n"               # a sentinel so the client knows we're finished

@app.post("/ask/stream")
async def ask_stream(req: AskRequest):
    return StreamingResponse(
        token_stream(req.question),
        media_type="text/event-stream",    # 🌊 the SSE content type
    )

Overwriting main.py


In [28]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

In [22]:
!pip install fastapi "uvicorn[standard]" streamlit requests groq langchain langchain-groq python-dotenv sse-starlette

In [23]:
!pip install pyngrok

In [29]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://walker-unsubduable-unsupplicatingly.ngrok-free.dev" -> "http://localhost:8000"


In [32]:
!uvicorn main:app --host 0.0.0.0 --port 8000

INFO:     Started server process [4565]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     128.185.168.213:0 - "GET /ask HTTP/1.1" 405 Method Not Allowed
INFO:     128.185.168.213:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     128.185.168.213:0 - "POST //ask HTTP/1.1" 404 Not Found
INFO:     128.185.168.213:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4565]
